In [ ]:
# UFC NLP + K-Means

import pandas as pd
import re
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

# Load data
df = pd.read_csv("complete_ufc_data.csv")
print(df.head())
print(df.columns)

# Select text column (change if needed)
text_col = df.select_dtypes(include="object").columns[0]

# Clean text
df["text"] = df[text_col].fillna("").str.lower()
df["text"] = df["text"].apply(lambda x: re.sub(r"[^a-z\s]", "", x))

# TF-IDF
tfidf = TfidfVectorizer(stop_words="english", max_features=3000)
X = tfidf.fit_transform(df["text"])

# K-Means
k = 4
model = KMeans(n_clusters=k, random_state=42, n_init=10)
df["cluster"] = model.fit_predict(X)

# Show clusters
print(df[["text", "cluster"]].head(20))

# Top words in each cluster
words = tfidf.get_feature_names_out()

for i in range(k):
    top = model.cluster_centers_[i].argsort()[-10:][::-1]
    print(f"\nCluster {i}:")
    print(", ".join(words[j] for j in top))

# PCA visualisation
pca = PCA(n_components=2)
points = pca.fit_transform(X.toarray())

plt.figure(figsize=(8, 5))
plt.scatter(points[:, 0], points[:, 1], c=df["cluster"])
plt.xlabel("PCA 1")
plt.ylabel("PCA 2")
plt.title("UFC K-Means Clusters")
plt.show()

# Save results
df.to_csv("ufc_kmeans_results.csv", index=False)

print("Done!")